In [27]:
"""Module for implementing logistic regression using scikit-learn.
This module standardizes and transforms the suicide study dataset, 
and applies logistic regression to analyze trends in suicide data 
from both 2023 and 2013-2022 periods."""

import pandas as pd
import numpy as np

import sys
from pathlib import Path
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

WORKSPACE_PATH = os.getenv("WORKSPACE_PATH")

# Add the parent directory to the system path
sys.path.append(str(WORKSPACE_PATH))

from src.config.config import DATA_DIR, RESULTS_DIR

In [28]:
# ================================================================================
# Data reading
# ================================================================================

# Define the path to the CSV file containing latent class analysis results
csv_file_path = RESULTS_DIR / "poLCA" / "Group_AG.csv"

# Read the CSV file into a DataFrame, specifying that it should not use low memory mode
lca_classes = pd.read_csv(
    csv_file_path, delimiter=",", low_memory=False, index_col=None
)

# Define the path to the encoded dataset CSV file
csv_file_path = DATA_DIR / "encoded" / "encoded_final_set.csv"

# Read the encoded dataset into a DataFrame
df_encoded = pd.read_csv(csv_file_path, delimiter=",", low_memory=False, index_col=None)

# Merge the encoded dataset with the latent class analysis results on the "ID" column
df_encoded = df_encoded.merge(lca_classes, on="ID", how="left")

# Extract and sort the unique groups from the merged DataFrame
groups = sorted(list(set(df_encoded["Group_AG"])))

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    mean_squared_error,
)


# Prepare encoder
encoder = OneHotEncoder(drop="first", sparse_output=False)

# Initialize results list
results_data = []

for group in groups:
    # Select group data
    group_data = df_encoded.loc[df_encoded["Group_AG"] == group].copy()

    # Ensure 'Predicted_Class_Group_AG' is categorical
    if group_data["Predicted_Class_Group_AG"].dtype != "category":
        group_data["Predicted_Class_Group_AG"] = group_data[
            "Predicted_Class_Group_AG"
        ].astype("category")

    # Prepare features and target
    X = encoder.fit_transform(group_data[["Predicted_Class_Group_AG"]])
    y = group_data["Fatal"].astype(int)

    # Handle edge case: Skip if too few samples
    if len(y) <= 1 or X.shape[1] == 0:
        continue

    # Fit logistic regression
    logreg_model = LogisticRegression(max_iter=1000, class_weight="balanced")
    logreg_model.fit(X, y)

    # Log-likelihood calculation
    log_probs = logreg_model.predict_proba(X)
    log_likelihood = np.sum(np.log(log_probs[np.arange(len(y)), y]))

    # Metrics calculation
    n_params = X.shape[1] + 1  # Number of coefficients + intercept
    aic = 2 * n_params - 2 * log_likelihood
    bic = np.log(len(y)) * n_params - 2 * log_likelihood

    # Predictive metrics calculation
    y_pred = logreg_model.predict(X)
    y_prob = logreg_model.predict_proba(X)[:, 1]
    precision = precision_score(y, y_pred, zero_division=0)
    recall = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    accuracy = accuracy_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_prob))

    # Store results
    for col, coef in zip(encoder.categories_[0][1:], logreg_model.coef_[0]):
        results_data.append(
            {
                "Group": group,
                "Variable": col,
                "Coefficient": coef,
                "Intercept": logreg_model.intercept_[0],
                "Log-Likelihood": log_likelihood,
                "AIC": aic,
                "BIC": bic,
                "Precision": precision,
                "Recall": recall,
                "F1-Score": f1,
                "Accuracy": accuracy,
                "RMSE": rmse,
            }
        )

# Create DataFrame with results
sklearn_results_df = pd.DataFrame(results_data)

In [33]:
sklearn_results_df

,Group,Variable,Coefficient,Intercept,Log-Likelihood,AIC,BIC,Precision,Recall,F1-Score,Accuracy,RMSE
0,00_18_F,2,2.387881,-2.124365,-1528.881055,3067.762110,3101.068578,0.440821,0.928753,0.597871,0.914978,0.264080
1,00_18_F,3,-0.074323,-2.124365,-1528.881055,3067.762110,3101.068578,0.440821,0.928753,0.597871,0.914978,0.264080
2,00_18_F,4,-0.624213,-2.124365,-1528.881055,3067.762110,3101.068578,0.440821,0.928753,0.597871,0.914978,0.264080
3,00_18_F,5,4.686854,-2.124365,-1528.881055,3067.762110,3101.068578,0.440821,0.928753,0.597871,0.914978,0.264080
4,00_18_M,2,-0.035960,-0.393598,-1680.958242,3371.916484,3401.794307,0.739130,0.428752,0.542698,0.803025,0.437384
5,00_18_M,3,-0.324156,-0.393598,-1680.958242,3371.916484,3401.794307,0.739130,0.428752,0.542698,0.803025,0.437384
6,00_18_M,4,0.137490,-0.393598,-1680.958242,3371.916484,3401.794307,0.739130,0.428752,0.542698,0.803025,0.437384
7,00_18_M,5,2.384537,-0.393598,-1680.958242,3371.916484,3401.794307,0.739130,0.428752,0.542698,0.803025,0.437384
8,19_34_F,2,-0.023747,-0.237316,-5811.193359,11632.386719,11667.676160,0.266667,0.329379,0.294724,0.736897,0.491329
9,19_34_F,3,0.187656,-0.237316,-5811.193359,11632.386719,11667.676160,0.266667,0.329379,0.294724,0.736897,0.491329
